# DX 704 Week 2 Project

This week's project will analyze fresh strawberry price data for a hypothetical "buy low, freeze, and sell high" business.
Strawberries show strong seasonality in their prices compared to other fruits.

![](https://ers.usda.gov/sites/default/files/_laserfiche/Charts/61401/oct14_finding_plattner_fig01.png)

Image source: https://www.ers.usda.gov/amber-waves/2014/october/seasonal-fresh-fruit-price-patterns-differ-across-commodities-the-case-of-strawberries-and-apples

You are considering a business where you buy strawberries when the prices are very low, carefully freeze them, even more carefully defrost them, and then sell them when the prices are high.
You will forecast strawberry price time series and then use them to tactically pick times to buy, freeze, and sell the strawberries.

The full project description, a template notebook, and raw data are available on GitHub at the following link.

https://github.com/bu-cds-dx704/dx704-project-02


### Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Backtest Strawberry Prices

Read the provided "strawberry-prices.tsv" with data from 2020 through 2025.
This data is based on data from the U.S. Bureau of Statistics, but transformed so the ground truth is not online.
https://fred.stlouisfed.org/series/APU0000711415

Use the data for 2020 through 2024 to predict monthly prices in 2025.
Spend some time to make sure you are happy with your methodology and prediction accuracy, since you will reuse the methodology to forecast 2026 next.
Save the 2025 backtest predictions as "strawberry-backtest.tsv" with columns month and price.

In [1]:
# Hint: beware of missing rows of data.
# The source is missing a few months!

In [2]:
import pandas as pd

# Read price data and make sure the month column is in a consistent format.
df = pd.read_csv("strawberry-prices.tsv", sep="\t")
df["month"] = pd.to_datetime(df["month"])
df = df.sort_values("month")

# Create a complete monthly series so the missing months are not lost.
all_months = pd.date_range(df["month"].min(), df["month"].max(), freq="MS")
full = pd.DataFrame({"month": all_months}).merge(df, on="month", how="left")
full["price"] = pd.to_numeric(full["price"], errors="coerce")

# Use the average price for each month of the year from 2020-2024 as the forecasting method.
train = full[full["month"].dt.year <= 2024].copy()
month_average = train.groupby(train["month"].dt.month)["price"].mean()

# Forecast all months of 2025 using the same month-of-year average.
backtest = pd.DataFrame({"month": pd.date_range("2025-01-01", "2025-12-01", freq="MS")})
backtest["price"] = backtest["month"].dt.month.map(month_average)
backtest["month"] = backtest["month"].dt.strftime("%Y-%m-01")
backtest.to_csv("strawberry-backtest.tsv", sep="\t", index=False)
backtest.head()

/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


,month,price
0,2025-01-01,4.5012
1,2025-02-01,4.1256
2,2025-03-01,3.6994
3,2025-04-01,3.8730
4,2025-05-01,3.4684


Please use the same format for the month column as in the training data, i.e. YYYY-MM-01.
The autograder may not be able to parse other formats.

Submit "strawberry-backtest.tsv" in Gradescope.

## Part 2: Backtest Errors

What are the mean and standard deviation of the residuals between your backtest predictions and the ground truth?

Write the mean and standard deviation to a file "backtest-accuracy.tsv" with two columns, mean and std.

In [3]:
# Compare the backtest predictions to the actual 2025 ground truth.
actual_2025 = full[full["month"].dt.year == 2025].dropna(subset=["price"]).copy()
actual_2025["predicted"] = actual_2025["month"].dt.month.map(month_average)
actual_2025["residual"] = actual_2025["price"] - actual_2025["predicted"]

mean_residual = actual_2025["residual"].mean()
std_residual = actual_2025["residual"].std(ddof=1)

accuracy = pd.DataFrame({
    "mean": [mean_residual],
    "std": [std_residual],
})
accuracy.to_csv("backtest-accuracy.tsv", sep="\t", index=False)

mean_residual, std_residual

(-0.06554000000000001, 0.1531480053920245)

Hint: If the mean residual in your backtest is not close to zero, then your model is likely missing a systematic change and you should go back to improve it.

Submit "backtest-accuracy.tsv" in Gradescope.

## Part 3: Forecast Strawberry Prices

Use all the data from 2020 through 2025 to predict monthly prices in 2026 using the same methodology from part 1.
Make a monthly forecast for each month of 2026 and save it as "strawberry-forecast.tsv" with columns for month and price.


In [4]:
# Forecast 2026 using the same month-of-year logic, now using all data through 2025.
all_data = full.dropna(subset=["price"]).copy()
forecast_month_average = all_data.groupby(all_data["month"].dt.month)["price"].mean()

forecast = pd.DataFrame({"month": pd.date_range("2026-01-01", "2026-12-01", freq="MS")})
forecast["price"] = forecast["month"].dt.month.map(forecast_month_average)
forecast["month"] = forecast["month"].dt.strftime("%Y-%m-01")
forecast.to_csv("strawberry-forecast.tsv", sep="\t", index=False)
forecast

,month,price
0,2026-01-01,4.515000
1,2026-02-01,4.117500
2,2026-03-01,3.644333
3,2026-04-01,3.802000
4,2026-05-01,3.459667
5,2026-06-01,3.211167
6,2026-07-01,3.179833
7,2026-08-01,3.456000
8,2026-09-01,3.619000
9,2026-10-01,3.884200


Submit "strawberry-forecast.tsv" in Gradescope.

## Part 4: Buy Low, Freeze and Sell High

Using your 2026 forecast, analyze the profit picking different pairs of months to buy and sell strawberries.
Maximize your profit assuming that it costs &dollar;0.20 per pint to freeze the strawberries, &dollar;0.10 per pint per month to store the frozen strawberries and there is a 10% price discount from selling previously frozen strawberries.
So, if you buy a pint of strawberies for &dollar;1, freeze them, and sell them for &dollar;2 three months after buying them, then the profit is &dollar;2 * 0.9 - &dollar;1 - &dollar;0.20 - &dollar;0.10 * 3 = &dollar;0.30 per pint.
To evaluate a given pair of months, assume that you can invest &dollar;1,000,000 to cover all costs, and that you buy as many pints of strawberries as possible.

Write the results of your analysis to a file "timings.tsv" with columns for the buy_month, sell_month, pints_purchased, and expected_profit.

In [5]:
# Evaluate all buy/sell month combinations for 2026 using the forecasted prices.
forecast_index = pd.to_datetime(forecast["month"])
prices = pd.Series(forecast["price"].values, index=forecast_index)

rows = []
for buy in forecast_index:
    for sell in forecast_index:
        if sell <= buy:
            continue
        months_gap = (sell.year - buy.year) * 12 + (sell.month - buy.month)
        buy_price = float(prices.loc[buy])
        sell_price = float(prices.loc[sell])
        per_pint_profit = sell_price * 0.9 - buy_price - 0.20 - 0.10 * months_gap
        total_cost = buy_price + 0.20 + 0.10 * months_gap
        pints_purchased = int(1_000_000 / total_cost)
        expected_profit = pints_purchased * per_pint_profit

        rows.append({
            "buy_month": buy.strftime("%Y-%m-01"),
            "sell_month": sell.strftime("%Y-%m-01"),
            "pints_purchased": pints_purchased,
            "expected_profit": expected_profit,
        })

results = pd.DataFrame(rows)

# Save the timing table.
results.to_csv("timings.tsv", sep="\t", index=False)
results.sort_values("expected_profit", ascending=False).head(10)

,buy_month,sell_month,pints_purchased,expected_profit
55,2026-07-01,2026-12-01,257743,144997.620367
50,2026-06-01,2026-12-01,249304,107508.194933
59,2026-08-01,2026-12-01,246548,95266.147200
62,2026-09-01,2026-12-01,242777,78514.081800
54,2026-07-01,2026-11-01,264561,48188.904280
64,2026-10-01,2026-12-01,233415,36926.253000
44,2026-05-01,2026-12-01,229375,18976.958333
49,2026-06-01,2026-11-01,255678,12991.851440
58,2026-08-01,2026-11-01,252780,1511.624400
61,2026-09-01,2026-11-01,248818,-14187.602360


Submit "timings.tsv" in Gradescope.

## Part 5: Strategy Check

What is the best profit scenario according to your previous timing analysis?
How much does that profit change if the sell price is off by one standard deviation from your backtest analysis?
(Variation in the sell price is more dangerous because you can see the buy price before fully committing.)

Write the results to a file "check.tsv" with columns `best_profit` and `one_std_profit`.
To be clear, `one_std_profit` should be the number of pints bought in your best profit scenario times your backtested standard deviation of the residual.
This represents the standard deviation in revenue when selling if you explicitly assume that you buy according to the best profit scenario and your backtest standard deviation is representative of the future prices.

In [6]:
best_row = results.sort_values("expected_profit", ascending=False).iloc[0]
best_profit = best_row["expected_profit"]
std = actual_2025["residual"].std(ddof=1)
one_std_profit = best_row["pints_purchased"] * std

check = pd.DataFrame({
    "best_profit": [best_profit],
    "one_std_profit": [one_std_profit],
})
check.to_csv("check.tsv", sep="\t", index=False)
check

,best_profit,one_std_profit
0,144997.620367,39472.826354


Submit "check.tsv" in Gradescope.

## Part 6: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgments are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

Submit "project.ipynb" in Gradescope.